# Moist Air Property Test

Functional tests for the moist-air property layer.

The notebook checks:
- availability of psychrometric and transport-property functions,
- practical moist-air states defined by `T/RH/p` and `T/W/p`,
- high-temperature states above water boiling temperature at given pressure,
- wet-process limits needed later by the wet economizer solver.

Core units:
- temperature: K
- pressure: Pa
- relative humidity: fraction `0.0 ... 1.0`
- humidity ratio: `kg_water/kg_dry_air`
- displayed humidity ratio: `g_water/kg_dry_air`
- moist-air enthalpy: `J/kg_dry_air`
- moist-air transport `cp`: `J/(kg_moist_air*K)`


In [ ]:
from pathlib import Path
import sys
import math

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)


In [ ]:
from core.psychrometrics import (
    humidity_ratio_to_g_per_kg_dry_air,
    max_relative_humidity_at_t_p,
    moist_air_state_from_t_rh,
    moist_air_state_from_t_w_g_per_kg_da,
    saturation_humidity_ratio,
    saturation_humidity_ratio_is_defined,
    saturation_vapor_pressure,
    wet_surface_process_limit,
)

from core.properties import (
    MoistAirTransportProvider,
    moist_air_transport_result_from_state,
    to_outside_fluid_props,
)

# Reference state: 20 degC, 50% RH, 1 atm.
T_ref = 293.15
RH_ref = 0.50
p_ref = 101325.0

air_ref = moist_air_state_from_t_rh(T=T_ref, RH=RH_ref, p=p_ref)
transport_ref = moist_air_transport_result_from_state(air_ref)

assert 0.0070 < air_ref.W < 0.0076, air_ref.W
assert 281.5 < air_ref.T_dew < 283.2, air_ref.T_dew
assert 38000.0 < air_ref.h < 39000.0, air_ref.h
assert 1.19 < air_ref.rho < 1.21, air_ref.rho

assert transport_ref.props.rho > 0.0
assert transport_ref.props.mu > 0.0
assert transport_ref.props.k > 0.0
assert transport_ref.props.cp > 0.0

print("Moist-air import and smoke test passed.")


In [ ]:
moist_air_cases = [
    {"case": "cold ventilation air", "T_C": 5.0, "RH": 0.80, "p_bar": 1.01325},
    {"case": "indoor reference", "T_C": 20.0, "RH": 0.50, "p_bar": 1.01325},
    {"case": "warm humid air", "T_C": 40.0, "RH": 0.70, "p_bar": 1.01325},
    {"case": "hot humid air", "T_C": 80.0, "RH": 0.20, "p_bar": 1.01325},
    {"case": "warm low-pressure air", "T_C": 60.0, "RH": 0.30, "p_bar": 0.80},

    # Above boiling at approximately atmospheric pressure.
    {"case": "over boiling check by RH", "T_C": 120.0, "RH": 0.30, "p_bar": 1.01325},
    {"case": "invalid high RH above boiling", "T_C": 120.0, "RH": 0.70, "p_bar": 1.01325},

    # Preferred input for hot gas / flue gas style cases.
    {"case": "over boiling by W", "T_C": 120.0, "W_g_per_kg_da": 120.0, "p_bar": 1.01325},
    {"case": "hot gas low moisture by W", "T_C": 180.0, "W_g_per_kg_da": 80.0, "p_bar": 1.01325},
]

rows = []

for case in moist_air_cases:
    T = case["T_C"] + 273.15
    p = case["p_bar"] * 1.0e5
    warning_codes = []

    try:
        if "RH" in case:
            input_mode = "T/RH/p"
            air = moist_air_state_from_t_rh(T=T, RH=case["RH"], p=p)
        elif "W_g_per_kg_da" in case:
            input_mode = "T/W_g_per_kg_da/p"
            air = moist_air_state_from_t_w_g_per_kg_da(
                T=T,
                W_g_per_kg_da=case["W_g_per_kg_da"],
                p=p,
            )
        else:
            raise ValueError("Case must define either RH or W_g_per_kg_da.")

        transport = moist_air_transport_result_from_state(air)
        warning_codes.extend(w.code for w in transport.warnings)

        if saturation_humidity_ratio_is_defined(T, p):
            W_sat = saturation_humidity_ratio(T, p)
            W_sat_g_per_kg_da = humidity_ratio_to_g_per_kg_dry_air(W_sat)
        else:
            W_sat_g_per_kg_da = math.nan
            warning_codes.append("SATURATION_STATE_NOT_DEFINED_AT_T_P")

        p_ws = saturation_vapor_pressure(T)

        row = {
            "case": case["case"],
            "input_mode": input_mode,
            "T_C": case["T_C"],
            "p_bar": case["p_bar"],
            "RH_input": case.get("RH", math.nan),
            "W_input_g_per_kg_da": case.get("W_g_per_kg_da", math.nan),
            "RH_calc": air.RH,
            "RH_max_at_T_p": max_relative_humidity_at_t_p(T, p),
            "p_ws_bar": p_ws / 1.0e5,
            "W_g_per_kg_da": humidity_ratio_to_g_per_kg_dry_air(air.W),
            "W_sat_g_per_kg_da": W_sat_g_per_kg_da,
            "T_dew_C": air.T_dew - 273.15,
            "h_kJ_per_kg_da": air.h / 1000.0,
            "rho_kg_m3": air.rho,
            "cp_J_kg_moist_K": transport.props.cp,
            "mu_Pa_s": transport.props.mu,
            "k_W_mK": transport.props.k,
            "Pr": transport.props.mu * transport.props.cp / transport.props.k,
            "warnings": ", ".join(sorted(set(warning_codes))),
            "error": "",
        }

    except Exception as exc:
        row = {
            "case": case["case"],
            "input_mode": "ERROR",
            "T_C": case["T_C"],
            "p_bar": case["p_bar"],
            "RH_input": case.get("RH", math.nan),
            "W_input_g_per_kg_da": case.get("W_g_per_kg_da", math.nan),
            "RH_calc": math.nan,
            "RH_max_at_T_p": max_relative_humidity_at_t_p(T, p),
            "p_ws_bar": saturation_vapor_pressure(T) / 1.0e5,
            "W_g_per_kg_da": math.nan,
            "W_sat_g_per_kg_da": math.nan,
            "T_dew_C": math.nan,
            "h_kJ_per_kg_da": math.nan,
            "rho_kg_m3": math.nan,
            "cp_J_kg_moist_K": math.nan,
            "mu_Pa_s": math.nan,
            "k_W_mK": math.nan,
            "Pr": math.nan,
            "warnings": "",
            "error": str(exc),
        }

    rows.append(row)

moist_air_df = pd.DataFrame(rows)

moist_air_df


In [ ]:
# Functional checks for the matrix.

indoor = moist_air_df.loc[moist_air_df["case"] == "indoor reference"].iloc[0]
assert indoor["input_mode"] == "T/RH/p"
assert indoor["error"] == ""
assert 7.0 < indoor["W_g_per_kg_da"] < 8.0
assert 9.0 < indoor["T_dew_C"] < 10.5
assert 38.0 < indoor["h_kJ_per_kg_da"] < 39.5
assert 1.19 < indoor["rho_kg_m3"] < 1.21

over_boiling_rh = moist_air_df.loc[moist_air_df["case"] == "over boiling check by RH"].iloc[0]
assert over_boiling_rh["error"] == ""
assert "SATURATION_STATE_NOT_DEFINED_AT_T_P" in over_boiling_rh["warnings"]
assert math.isnan(over_boiling_rh["W_sat_g_per_kg_da"])

over_boiling_w = moist_air_df.loc[moist_air_df["case"] == "over boiling by W"].iloc[0]
assert over_boiling_w["error"] == ""
assert over_boiling_w["input_mode"] == "T/W_g_per_kg_da/p"
assert "SATURATION_STATE_NOT_DEFINED_AT_T_P" in over_boiling_w["warnings"]
assert math.isnan(over_boiling_w["W_sat_g_per_kg_da"])

invalid_high_rh = moist_air_df.loc[moist_air_df["case"] == "invalid high RH above boiling"].iloc[0]
assert invalid_high_rh["input_mode"] == "ERROR"
assert "instead of RH" in invalid_high_rh["error"]

print("Moist-air functional matrix test passed.")


In [ ]:
wet_process_cases = [
    {"case": "dry surface above dew point", "T_air_C": 20.0, "RH": 0.50, "p_bar": 1.01325, "T_surface_C": 12.0},
    {"case": "wet surface below dew point", "T_air_C": 20.0, "RH": 0.50, "p_bar": 1.01325, "T_surface_C": 7.0},
    {"case": "frost risk surface", "T_air_C": 20.0, "RH": 0.50, "p_bar": 1.01325, "T_surface_C": -3.0},
    {"case": "hot humid gas to cool wall", "T_air_C": 80.0, "RH": 0.20, "p_bar": 1.01325, "T_surface_C": 45.0},
]

wet_rows = []

for case in wet_process_cases:
    T_air = case["T_air_C"] + 273.15
    T_surface = case["T_surface_C"] + 273.15
    p = case["p_bar"] * 1.0e5

    air = moist_air_state_from_t_rh(T=T_air, RH=case["RH"], p=p)
    result = wet_surface_process_limit(air=air, T_surface=T_surface)
    warning_codes = [w.code for w in result.warnings]

    wet_rows.append(
        {
            "case": case["case"],
            "T_air_C": case["T_air_C"],
            "RH": case["RH"],
            "p_bar": case["p_bar"],
            "T_surface_C": case["T_surface_C"],
            "T_dew_C": air.T_dew - 273.15,
            "will_condense": result.condensation.will_condense,
            "dew_point_margin_K": result.condensation.dew_point_margin,
            "W_bulk_g_per_kg_da": humidity_ratio_to_g_per_kg_dry_air(air.W),
            "W_surface_sat_g_per_kg_da": humidity_ratio_to_g_per_kg_dry_air(
                result.surface_saturated_state.W
            ),
            "condensable_water_g_per_kg_da": result.condensable_water * 1000.0,
            "enthalpy_drop_kJ_per_kg_da": result.enthalpy_drop / 1000.0,
            "warnings": ", ".join(sorted(set(warning_codes))),
        }
    )

wet_process_df = pd.DataFrame(wet_rows)

wet_process_df


In [ ]:
dry_case = wet_process_df.loc[wet_process_df["case"] == "dry surface above dew point"].iloc[0]
wet_case = wet_process_df.loc[wet_process_df["case"] == "wet surface below dew point"].iloc[0]
frost_case = wet_process_df.loc[wet_process_df["case"] == "frost risk surface"].iloc[0]

assert not bool(dry_case["will_condense"])
assert dry_case["condensable_water_g_per_kg_da"] == 0.0

assert bool(wet_case["will_condense"])
assert wet_case["condensable_water_g_per_kg_da"] > 0.0
assert wet_case["enthalpy_drop_kJ_per_kg_da"] > 0.0

assert bool(frost_case["will_condense"])
assert "SURFACE_BELOW_FREEZING" in frost_case["warnings"]

print("Wet-process functional matrix test passed.")


In [ ]:
provider = MoistAirTransportProvider.from_t_rh(
    T=293.15,
    RH=0.50,
    p=101325.0,
)

provider_cases = [
    {"T_C": 10.0, "p_bar": 1.01325},
    {"T_C": 20.0, "p_bar": 1.01325},
    {"T_C": 40.0, "p_bar": 1.01325},
    {"T_C": 80.0, "p_bar": 1.01325},
]

provider_rows = []

for case in provider_cases:
    T = case["T_C"] + 273.15
    p = case["p_bar"] * 1.0e5

    props = provider.at(T=T, p=p)
    outside_props = to_outside_fluid_props(props)

    provider_rows.append(
        {
            "T_C": case["T_C"],
            "p_bar": case["p_bar"],
            "rho_kg_m3": props.rho,
            "mu_Pa_s": props.mu,
            "k_W_mK": props.k,
            "cp_J_kg_moist_K": props.cp,
            "Pr": props.mu * props.cp / props.k,
            "outside_props_ok": (
                outside_props.rho == props.rho
                and outside_props.mu == props.mu
                and outside_props.k == props.k
                and outside_props.cp == props.cp
            ),
        }
    )

provider_df = pd.DataFrame(provider_rows)

assert provider_df["outside_props_ok"].all()
assert (provider_df["rho_kg_m3"] > 0.0).all()
assert (provider_df["mu_Pa_s"] > 0.0).all()
assert (provider_df["k_W_mK"] > 0.0).all()
assert (provider_df["cp_J_kg_moist_K"] > 0.0).all()

provider_df
